## Import

In [ ]:
!pip install rasterio

In [2]:
import rasterio

## How to open dataset

In [3]:
dataset = rasterio.open('data/23-05-2020.tiff')

In [ ]:
dict(
    dataset.profile
)

In [ ]:
dataset.bounds

## How to read bands

In [6]:
band4 = dataset.read(4)
band8 = dataset.read(8)


In [ ]:
band4

## How to write

In [8]:
with rasterio.open(
        'tmp.tif',
        'w',
        driver='GTiff',
        height=band4.shape[0],
        width=band4.shape[1],
        count=1,
        dtype=band4.dtype,
        crs=dataset.crs,
        transform=dataset.transform,
        nodata=dataset.nodata
    ) as file:

    file.write(band4, 1)

In [9]:
new_meta = dataset.meta.copy()
new_meta.update({
        'count': 1
    })

with rasterio.open(
        'tmp2.tif',
        'w',
        **new_meta
    ) as file:

    file.write(band4, 1)

## Reproject

In [10]:
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

In [11]:
dst_crs = 'EPSG:4326'

with rasterio.open('tmp.tif') as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open('tmp_reprojected.tif', 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=src.read(i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)

## Resample

In [12]:
import rasterio
from rasterio.enums import Resampling


In [13]:
upscale_factor = 2

with rasterio.open("tmp.tif") as dataset:

    # resample data to target shape
    data = dataset.read(
        out_shape=(
            dataset.count,
            int(dataset.height * upscale_factor),
            int(dataset.width * upscale_factor)
        ),
        resampling=Resampling.bilinear
    )

    # scale image transform
    transform = dataset.transform * dataset.transform.scale(
        (dataset.width / data.shape[-1]),
        (dataset.height / data.shape[-2])
    )

In [ ]:
Resampling._member_names_


In [15]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(data[0])

## Calculate indexes

In [17]:
src = rasterio.open('data/23-05-2020.tiff')

In [18]:
b4 = src.read(4)
b8 = src.read(8)

NIR = (b8 - b4) / (b4 + b8)

In [ ]:
plt.imshow(NIR)
plt.colorbar()

### Tasks

- Calculate any 5 indexes (https://www.indexdatabase.de/db/is.php?sensor_id=96)
- Save each index as a separate file
- Reproject each file to another CRS (For example, 4326)

In [20]:
# TODO